# Reto VI — Text-to-SQL corporativo con memoria, evaluación y aprendizaje

Este notebook implementa una **versión mínima, funcional y explicable** del reto:

1. **Carga el CSV** de empleados en memoria.
2. **Usa DuckDB** para ejecutar SQL directamente sobre los datos.
3. Define un **glosario de reglas HR** que sirve como ground truth.
4. Construye un **planner simple** que mapea una pregunta a una métrica del glosario.
5. Ejecuta el SQL con un **executer**.
6. Guarda y recupera **casos previos** como memoria tipo *memento*.
7. Evalúa si el SQL cumple exactamente con el glosario y asigna una **recompensa binaria**.
8. Entrena una **red neuronal mínima** con **Adam** usando esa recompensa.
9. Expone una **UI interactiva en Jupyter** usando `ipywidgets`.

> La solución está simplificada para que sea fácil de explicar y suficiente para demostrar cada parte pedida por el reto.

## Paso 1: Setup e ingesta del CSV en DuckDB

En esta sección:
- instalamos dependencias mínimas,
- cargamos el CSV con `pandas`,
- lo registramos en DuckDB,
- y creamos una tabla en memoria para poder hacer consultas SQL.

In [38]:
# Instalamos las librerías mínimas para esta primera parte.
# - duckdb: motor SQL embebido, muy útil para analytics y notebooks.
# - pandas: lectura y manipulación del CSV.
!pip install -q duckdb pandas

In [39]:
# Importamos las librerías que vamos a usar en la carga inicial de datos.
import duckdb
import pandas as pd

In [40]:
# =========================
# 1) Carga del dataset
# =========================
# Leemos el archivo CSV de empleados.
# Aquí asumimos que el notebook vive en una carpeta y el CSV en ../data/data.csv.
# Si cambias la ubicación del archivo, solo ajusta esta ruta.
df = pd.read_csv("../data/data.csv")

# Mostramos forma y columnas para validar rápidamente que el dataset cargó bien.
print("Shape:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

Shape: (1470, 35)

Columnas:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [41]:
# =========================
# 2) Conexión a DuckDB
# =========================
# Creamos una base de datos *en memoria*.
# Esto significa que vive solo durante la ejecución del notebook:
# es ideal para demos y retos porque no requiere archivos externos.
con = duckdb.connect(database=':memory:')

# Registramos el DataFrame como una vista temporal dentro de DuckDB.
# A partir de aquí podemos referenciar "employees_df" en SQL.
con.register("employees_df", df)

In [42]:
# =========================
# 3) Crear tabla SQL real
# =========================
# Convertimos la vista temporal en una tabla física dentro de DuckDB.
# Esto nos deja una tabla llamada "employees" sobre la cual el agente hará consultas.
con.execute("""
    CREATE TABLE employees AS
    SELECT * FROM employees_df
""")

In [43]:
# Validación rápida: contamos cuántas filas quedaron en la tabla SQL.
print("\nConteo de filas en DuckDB:")
print(con.execute("SELECT COUNT(*) AS total FROM employees").fetchdf())


Conteo de filas en DuckDB:
   total
0   1470


### Paso 1.1: Validar columnas del glosario

Antes de construir reglas, verificamos que el dataset sí contenga
las columnas necesarias para definir las métricas del reto.

In [44]:
# Lista de columnas mínimas que exige el glosario del reto.
# Si alguna falta, no podremos construir correctamente las reglas HR.
required_columns = [
    "PerformanceRating",
    "MonthlyIncome",
    "TrainingTimesLastYear",
    "YearsAtCompany",
    "Attrition",
    "YearsInCurrentRole",
    "YearsSinceLastPromotion"
]

In [45]:
# Buscamos qué columnas faltan comparando el glosario contra el DataFrame real.
missing = [c for c in required_columns if c not in df.columns]

if missing:
    print("Faltan columnas:", missing)
else:
    print("OK. Están todas las columnas necesarias para el glosario.")

OK. Están todas las columnas necesarias para el glosario.


In [46]:
# DuckDB ya responde a SQL, así que hacemos una consulta pequeña de inspección.
# Esto nos ayuda a comprobar:
# - que la tabla existe,
# - que la sintaxis SQL está funcionando,
# - y que algunas columnas importantes ya son consultables.
con.execute("""
    SELECT 
        Attrition, 
        PerformanceRating, 
        MonthlyIncome,
        YearsAtCompany
    FROM employees
    LIMIT 5
""").fetchdf()

,Attrition,PerformanceRating,MonthlyIncome,YearsAtCompany
0,Yes,3,5993,6
1,No,4,5130,10
2,Yes,3,2090,0
3,No,3,2909,8
4,No,3,3468,2


## Paso 2: Glosario de HR como reglas SQL

En vez de "inventar" las métricas, definimos un glosario centralizado
que actúa como fuente de verdad. Esto es clave porque después el evaluador
comparará el SQL generado contra estas reglas.

In [47]:
# Cargamos el glosario desde un archivo JSON.
# Esto tiene dos ventajas:
# 1) separa la lógica de negocio del código,
# 2) facilita crecer a más reglas sin tocar el planner.
import json

RULES_DATASET_PATH = "../data/rules.json"

with open(RULES_DATASET_PATH, "r", encoding="utf-8") as f:
    rulesDataset = json.load(f)

# Mostrar el contenido puede ayudar a depurar o documentar.
rulesDataset

{'flight_risk': {'label': 'Flight Risk',
  'sql_condition': 'PerformanceRating = 4 AND MonthlyIncome < 5000 AND TrainingTimesLastYear = 0'},
 'rising_star': {'label': 'Rising Star',
  'sql_condition': "YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'"},
 'stagnant_role': {'label': 'Stagnant Role',
  'sql_condition': 'YearsInCurrentRole > 5 AND YearsSinceLastPromotion > 5'}}

### Paso 2.1: SQL correcto para la pregunta de prueba

Construimos manualmente el SQL de la métrica **Rising Star**
para tener una referencia base y validar que el resultado del dataset tiene sentido.

In [48]:
# SQL de referencia para la métrica "rising_star".
# Lo construimos desde el glosario, no a mano, para mantener consistencia.
sql_rising_star = f"""
SELECT COUNT(*) AS total_rising_stars
FROM employees
WHERE {rulesDataset['rising_star']['sql_condition']}
"""

print(sql_rising_star)
result = con.execute(sql_rising_star).fetchdf()
result


SELECT COUNT(*) AS total_rising_stars
FROM employees
WHERE YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'



,total_rising_stars
0,30


### Paso 2.2: Inspección de registros que cumplen Rising Star

No solo contamos resultados; también vemos ejemplos concretos de empleados
que sí cumplen la regla para validar visualmente que el filtro es correcto.

In [49]:
# Mostramos algunos empleados que cumplen la regla "rising_star".
# Esto sirve como sanity check del filtro.
con.execute(f"""
SELECT 
    EmployeeNumber,
    YearsAtCompany,
    PerformanceRating,
    Attrition,
    MonthlyIncome
FROM employees
WHERE {rulesDataset['rising_star']['sql_condition']}
LIMIT 10
""").fetchdf()

,EmployeeNumber,YearsAtCompany,PerformanceRating,Attrition,MonthlyIncome
0,10,1,4,No,2670
1,11,1,4,No,2693
2,61,1,4,No,3022
3,155,1,4,No,2835
4,214,1,4,No,3346
5,217,2,4,No,2323
6,473,0,4,No,12504
7,477,1,4,No,6781
8,521,1,4,No,3448
9,544,2,4,No,2654


## Paso 3: Planner con LLM usando LangChain

A partir de este punto, el **planner principal** ya no es una función por reglas,
sino un **LLM** configurado con LangChain y salida estructurada.

La idea es:

1. el usuario escribe una pregunta en lenguaje natural,
2. el planner la mapea a una de las métricas del glosario,
3. decide si conviene consultar memoria,
4. y deja la ejecución SQL para componentes posteriores.

Esto alinea mejor el notebook con el enfoque del reto: **planner + executer + memoria + evaluación + aprendizaje**.

In [50]:
# Instalamos las dependencias necesarias para el planner con LLM.
# - langchain: framework para orquestar el agente y sus herramientas.
# - langchain-openai: integración de LangChain con modelos de OpenAI.
# - pydantic: define el esquema estructurado que el planner debe devolver.
!pip install -q -U langchain langchain-openai pydantic

In [51]:
# Imports base del planner y de la integración con herramientas.
import os
from typing import Optional, Literal
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.tools import tool

### Paso 3.1: Autenticación

Pedimos la API key de OpenAI en tiempo de ejecución y la guardamos también
en la variable de entorno `OPENAI_API_KEY`, porque LangChain la usa para
crear el planner basado en LLM.

In [52]:
# Solicitamos la API key de OpenAI de forma segura.
# La guardamos:
# 1) en una variable local para este notebook,
# 2) en el entorno para que LangChain la detecte.
import getpass
from openai import OpenAI

api_key = getpass.getpass("🔑 Introduce tu OpenAI API Key: ").strip()

# Cliente oficial de OpenAI.
# No es obligatorio para LangChain, pero puede ser útil si luego quieres
# hacer llamadas directas al SDK oficial.
client = OpenAI(api_key=api_key)

# LangChain suele leer la credencial desde esta variable de entorno.
os.environ["OPENAI_API_KEY"] = api_key

### Paso 3.2: Salida estructurada del planner

En vez de dejar que el LLM responda texto libre, le pedimos un objeto estructurado.
Esto hace más predecible la orquestación porque el planner debe devolver:

- `metric_key`: la métrica del glosario,
- `use_memory`: si conviene consultar memoria previa,
- `reasoning`: explicación breve de la decisión.

In [73]:
# Definimos el esquema exacto que debe devolver el planner.
# Usamos Pydantic porque LangChain puede mapear la salida del LLM
# a este modelo estructurado.
class PlannerOutput(BaseModel):
    metric_key: Optional[Literal["rising_star", "flight_risk", "stagnant_role"]] = Field(
        default=None,
        description="Clave exacta de la métrica del glosario HR."
    )
    use_memory: bool = Field(
        default=True,
        description="Indica si conviene consultar memoria de casos previos."
    )
    reasoning: str = Field(
        description="Breve explicación de por qué eligió esa métrica."
    )

# Convertimos el glosario en texto para incrustarlo en el prompt del planner.
# Así el modelo no inventa reglas: solo puede elegir entre las que existen.
def glossary_as_text() -> str:
    rows = []
    for key, item in rulesDataset.items():
        rows.append(f"- {key}: {item['label']} => {item['sql_condition']}")
    return "\n".join(rows)

# Prompt del planner: delimita claramente su responsabilidad.
# Importante:
# - el planner NO genera SQL,
# - el planner NO consulta DuckDB,
# - el planner SOLO decide la métrica y si conviene usar memoria.
PLANNER_SYSTEM_PROMPT = f"""
Eres el planner de un sistema Text-to-SQL corporativo de HR.

Tu trabajo es:
1. Leer la pregunta del usuario.
2. Mapearla a UNA métrica del glosario, si existe.
3. Indicar si se debe usar memoria de casos previos.
4. Responder SOLO en el formato estructurado pedido.

Glosario disponible:
{glossary_as_text()}

Reglas:
- SOLO puedes elegir una de estas metric_key:
  - rising_star
  - flight_risk
  - stagnant_role
- Si la pregunta no corresponde a ninguna métrica, devuelve metric_key = null.
- No generes SQL aquí.
- No inventes métricas fuera del glosario.
- Si metric_key es válida, devuelve SIEMPRE use_memory=true.
- Solo devuelve use_memory=false cuando metric_key = null.
"""

# Creamos el planner agent.
# Nota:
# usamos un modelo ligero para que el notebook sea más rápido y económico.
planner_agent = create_agent(
    model="gpt-4.1-mini",
    tools=[],
    system_prompt=PLANNER_SYSTEM_PROMPT,
    response_format=PlannerOutput,
)

# Función contenedora para invocar el planner.
# Devuelve directamente un objeto PlannerOutput.
def run_planner(question: str) -> PlannerOutput:
    result = planner_agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    return result["structured_response"]

### Paso 3.3: Utilidades auxiliares

Aunque el planner principal ya es un LLM, dejamos una función **heurística auxiliar**
para dos tareas internas muy simples:

- recuperar memoria previa sin volver a llamar al LLM,
- construir features binarias para la red neuronal.

Esta heurística **no reemplaza** al planner; solo evita llamadas extras al modelo
en partes donde no hacen falta.

In [54]:
# Normalización simple de texto.
# La usamos en funciones auxiliares para comparar frases de manera robusta.
def normalize_text(text: str) -> str:
    return text.strip().lower()

# Heurística auxiliar para detectar una métrica desde patrones.
# OJO:
# esta función NO es el planner oficial del sistema.
# Solo se usa como ayuda barata para:
# - lookup de memoria,
# - extracción de features de la red neuronal.
def heuristic_metric_from_question(question: str) -> str | None:
    q = normalize_text(question)

    rising_star_patterns = [
        "estrellas en ascenso",
        "estrella en ascenso",
        "rising star",
        "rising stars",
    ]
    flight_risk_patterns = [
        "flight risk",
        "riesgo de fuga",
    ]
    stagnant_role_patterns = [
        "stagnant role",
        "rol estancado",
        "puesto estancado",
    ]

    if any(p in q for p in rising_star_patterns):
        return "rising_star"
    if any(p in q for p in flight_risk_patterns):
        return "flight_risk"
    if any(p in q for p in stagnant_role_patterns):
        return "stagnant_role"

    return None

## Paso 4: Memoria tipo memento

Ahora agregamos una memoria muy simple de casos previos.
Cada caso guardado representa una experiencia del agente:

- pregunta original,
- métrica detectada,
- SQL usado,
- recompensa obtenida,
- vista resumida del resultado.

La memoria nos permite reutilizar casos exitosos cuando llega una pregunta equivalente
o muy parecida.

In [55]:
# Lista en memoria donde almacenamos casos previos.
# Vive solo durante la sesión del notebook.
memory_cases = []

# Guarda un caso resuelto en memoria.
def save_case(question: str, metric_key: str, sql: str, reward: float, result_preview=None):
    memory_cases.append({
        "question": question,
        "metric_key": metric_key,
        "sql": sql,
        "reward": reward,
        "result_preview": result_preview
    })

# Recupera el caso exitoso más reciente para la misma métrica.
# Para no hacer una llamada extra al LLM en cada lookup, usamos la heurística auxiliar.
def retrieve_similar_case(question: str):
    metric_key = heuristic_metric_from_question(question)
    if metric_key is None:
        return None

    for case in reversed(memory_cases):
        if case["metric_key"] == metric_key and case["reward"] == 1.0:
            return case

    return None

## Paso 5: Generación de SQL, tools y evaluación

En esta etapa definimos tres piezas clave:

1. una función que genera el SQL a partir del glosario,
2. tools de LangChain para memoria, ejecución y evaluación,
3. un evaluador estricto que compara el SQL producido contra la regla oficial.

Con esto ya tenemos el bloque **executer + evaluator** del sistema.

In [56]:
# Importamos utilidades JSON y expresiones regulares.
# - json: serialización simple para tools de LangChain.
# - re: normalización/comparación de SQL.
import json
import re

# Construye el SQL de conteo a partir de la métrica detectada.
# El glosario sigue siendo la fuente de verdad del filtro.
def build_count_sql(metric_key: str) -> str:
    if metric_key not in rulesDataset:
        raise ValueError(f"Métrica no soportada: {metric_key}")

    condition = rulesDataset[metric_key]["sql_condition"]

    return f"""
    SELECT COUNT(*) AS total
    FROM employees
    WHERE {condition}
    """

# Normalización básica de SQL para poder comparar expresiones
# ignorando espacios, mayúsculas y comillas dobles/simples.
def normalize_sql_text(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = text.replace('"', "'")
    return text

# Extrae únicamente la cláusula WHERE del SQL generado.
# Así comparamos la lógica del filtro, no el resto del SELECT.
def extract_where_clause(sql: str) -> str:
    sql_norm = normalize_sql_text(sql)
    match = re.search(r"where (.+)", sql_norm, re.IGNORECASE)
    if not match:
        return ""
    return match.group(1).strip().rstrip(";").strip()

# Evaluador estricto:
# compara el WHERE generado contra la condición oficial del glosario.
def evaluate_sql_against_glossary(metric_key: str, sql: str) -> dict:
    if metric_key not in rulesDataset:
        return {
            "metric_key": metric_key,
            "reward": 0.0,
            "is_correct": False,
            "reason": "La métrica no existe en el glosario."
        }

    expected_condition = rulesDataset[metric_key]["sql_condition"]
    expected_norm = normalize_sql_text(expected_condition)
    generated_where = extract_where_clause(sql)
    generated_norm = normalize_sql_text(generated_where)

    is_correct = generated_norm == expected_norm
    reward = 1.0 if is_correct else 0.0

    return {
        "metric_key": metric_key,
        "expected_condition": expected_condition,
        "generated_where": generated_where,
        "is_correct": is_correct,
        "reward": reward,
        "reason": "SQL cumple exactamente con el glosario." if is_correct else "SQL no cumple exactamente con el glosario."
    }

# ---------------------------
# Tools del agente
# ---------------------------

@tool
def memory_tool(question: str) -> str:
    """Recupera un caso previo similar desde la memoria memento."""
    case = retrieve_similar_case(question)
    if case is None:
        return json.dumps({"found": False, "case": None}, ensure_ascii=False)

    return json.dumps({
        "found": True,
        "case": case
    }, ensure_ascii=False)

@tool
def sql_executor_tool(metric_key: str) -> str:
    """Genera y ejecuta el SQL de conteo para una metric_key válida."""
    if metric_key not in rulesDataset:
        return json.dumps({
            "ok": False,
            "error": f"Métrica no soportada: {metric_key}"
        }, ensure_ascii=False)

    sql = build_count_sql(metric_key)
    result_df = con.execute(sql).fetchdf()

    return json.dumps({
        "ok": True,
        "metric_key": metric_key,
        "sql": sql,
        "result": result_df.to_dict(orient="records")
    }, ensure_ascii=False)

@tool
def evaluator_tool(metric_key: str, sql: str) -> str:
    """Evalúa si el SQL cumple exactamente con el glosario HR."""
    evaluation = evaluate_sql_against_glossary(metric_key, sql)
    return json.dumps(evaluation, ensure_ascii=False)

## Paso 6: Red neuronal mínima + optimizador Adam

La red no genera SQL ni sustituye al planner.
Su papel en esta demo es aprender una **señal de confianza/recompensa**
a partir de features simples de la pregunta y del contexto de memoria.

In [57]:
# Instalamos PyTorch para construir la red neuronal mínima.
!pip install -q torch

In [58]:
# Imports de PyTorch.
import torch
import torch.nn as nn
import torch.optim as optim

### Paso 6.1: Ingeniería de features

Representamos cada pregunta con 4 señales binarias:

1. si menciona Rising Star,
2. si menciona Flight Risk,
3. si menciona Stagnant Role,
4. si ya existe memoria previa útil.

Es una representación muy simple, pero suficiente para demostrar
la actualización de pesos con Adam.

In [59]:
# Convierte una pregunta en un tensor de 4 features binarias.
def question_to_features(question: str):
    q = normalize_text(question)

    rising_star_patterns = ["estrellas en ascenso", "estrella en ascenso", "rising star", "rising stars"]
    flight_risk_patterns = ["flight risk", "riesgo de fuga"]
    stagnant_role_patterns = ["stagnant role", "rol estancado", "puesto estancado"]

    has_rising = 1.0 if any(p in q for p in rising_star_patterns) else 0.0
    has_flight = 1.0 if any(p in q for p in flight_risk_patterns) else 0.0
    has_stagnant = 1.0 if any(p in q for p in stagnant_role_patterns) else 0.0
    has_memory = 1.0 if retrieve_similar_case(question) is not None else 0.0

    return torch.tensor([has_rising, has_flight, has_stagnant, has_memory], dtype=torch.float32)

### Paso 6.2: Definir la red

Usamos una red muy pequeña:

- entrada: 4 features,
- capa oculta: 8 neuronas,
- salida: 1 valor entre 0 y 1.

In [60]:
# Red neuronal mínima para estimar un score de recompensa.
class RewardNet(nn.Module):
    def __init__(self, input_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Instanciamos red, optimizador Adam y pérdida binaria.
reward_net = RewardNet()
optimizer = optim.Adam(reward_net.parameters(), lr=0.01)
criterion = nn.BCELoss()

reward_net

RewardNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=1, bias=True)
    (3): Sigmoid()
  )
)

### Paso 6.3: Entrenamiento, predicción y orquestador principal

Aquí conectamos todo el pipeline:

1. el planner LLM decide la métrica,
2. opcionalmente se consulta memoria,
3. se ejecuta SQL en DuckDB,
4. el evaluador asigna recompensa,
5. la red se actualiza con Adam,
6. el caso queda guardado en memoria.

In [61]:
# Entrena la red con una sola observación.
def train_reward_net(question: str, reward: float):
    reward_net.train()

    x = question_to_features(question).unsqueeze(0)
    y = torch.tensor([[reward]], dtype=torch.float32)

    pred = reward_net(x)
    loss = criterion(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return {
        "pred_before_update": float(pred.detach().item()),
        "loss": float(loss.detach().item()),
        "target_reward": reward
    }

# Inferencia pura: estima el score sin actualizar pesos.
def predict_reward_score(question: str):
    reward_net.eval()
    with torch.no_grad():
        x = question_to_features(question).unsqueeze(0)
        pred = reward_net(x)
    return float(pred.item())

# Orquestador principal del sistema.
def ask_agent_with_learning(question: str):
    # 1) Predicción previa de la red.
    predicted_score_before = predict_reward_score(question)

    # 2) Planner LLM.
    planner_output = run_planner(question)
    metric_key = planner_output.metric_key

    if metric_key is None:
        reward = 0.0
        training_info = train_reward_net(question, reward)

        return {
            "question": question,
            "metric_key": None,
            "used_memory": False,
            "planner_reasoning": planner_output.reasoning,
            "sql": None,
            "result": None,
            "evaluation": {
                "reward": reward,
                "is_correct": False,
                "reason": "El planner no pudo mapear la pregunta a una métrica válida del glosario."
            },
            "predicted_score_before": predicted_score_before,
            "training_info": training_info,
        }

    # 3) Memoria opcional.
    retrieved_case = None
    used_memory = False

    if planner_output.use_memory:
        memory_raw = memory_tool.invoke({"question": question})
        memory_data = json.loads(memory_raw)

        if memory_data.get("found"):
            retrieved_case = memory_data["case"]
            used_memory = True

    # 4) SQL + ejecución.
    if retrieved_case is not None and retrieved_case.get("reward") == 1.0:
        sql = retrieved_case["sql"]
        result_df = con.execute(sql).fetchdf()
    else:
        executor_raw = sql_executor_tool.invoke({"metric_key": metric_key})
        executor_payload = json.loads(executor_raw)

        if not executor_payload.get("ok", False):
            reward = 0.0
            training_info = train_reward_net(question, reward)
            return {
                "question": question,
                "metric_key": metric_key,
                "used_memory": used_memory,
                "planner_reasoning": planner_output.reasoning,
                "sql": None,
                "result": None,
                "evaluation": {
                    "reward": reward,
                    "is_correct": False,
                    "reason": executor_payload.get("error", "Error en SQL executor tool.")
                },
                "predicted_score_before": predicted_score_before,
                "training_info": training_info,
            }

        sql = executor_payload["sql"]
        result_df = pd.DataFrame(executor_payload["result"])

    # 5) Evaluación estricta.
    evaluation_raw = evaluator_tool.invoke({
        "metric_key": metric_key,
        "sql": sql
    })
    evaluation = json.loads(evaluation_raw)
    reward = evaluation["reward"]

    # 6) Aprendizaje.
    training_info = train_reward_net(question, reward)

    # 7) Persistencia en memoria.
    save_case(
        question=question,
        metric_key=metric_key,
        sql=sql,
        reward=reward,
        result_preview=result_df.to_dict(orient="records")
    )

    return {
        "question": question,
        "metric_key": metric_key,
        "used_memory": used_memory,
        "planner_reasoning": planner_output.reasoning,
        "sql": sql,
        "result": result_df,
        "evaluation": evaluation,
        "predicted_score_before": predicted_score_before,
        "training_info": training_info,
    }

## Paso 7: UI interactiva dentro de Jupyter

No usamos Streamlit.
La interfaz vive dentro del notebook y permite:

- escribir una pregunta,
- ejecutar con **Enter** o con botón,
- limpiar el input después de enviar,
- detener la interfaz cuando ya no se quiera usar.

In [62]:
# ipywidgets permite crear la UI interactiva dentro de Jupyter.
!pip install -q ipywidgets

In [63]:
# Imports para widgets y render.
import ipywidgets as widgets
from IPython.display import display, clear_output

### Paso 7.1: Demo

Esta función es útil para depuración o para explicar el flujo completo
sin usar la interfaz interactiva.

In [64]:
# Ejecuta el pipeline completo y muestra el detalle en consola.
def run_demo(question: str):
    response = ask_agent_with_learning(question)

    print("=" * 80)
    print("PREGUNTA:")
    print(response["question"])

    print("\nMÉTRICA DETECTADA:")
    print(response["metric_key"])

    print("\nPLANNER REASONING:")
    print(response["planner_reasoning"])

    print("\n¿USÓ MEMORIA?")
    print(response["used_memory"])

    print("\nSQL GENERADO:")
    print(response["sql"])

    print("\nEVALUACIÓN:")
    for k, v in response["evaluation"].items():
        print(f"{k}: {v}")

    print("\nRED NEURONAL:")
    print("Predicted score before update:", response["predicted_score_before"])
    print("Training info:", response["training_info"])

    print("\nRESULTADO:")
    display(response["result"])

    print("=" * 80)
    return response

In [67]:
run_demo("¿Cuántos empleados son estrellas en ascenso?")

PREGUNTA:
¿Cuántos empleados son estrellas en ascenso?

MÉTRICA DETECTADA:
rising_star

PLANNER REASONING:
La pregunta se refiere directamente a empleados que son 'estrellas en ascenso', lo cual coincide exactamente con la métrica 'rising_star' del glosario.

¿USÓ MEMORIA?
False

SQL GENERADO:

    SELECT COUNT(*) AS total
    FROM employees
    WHERE YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'
    

EVALUACIÓN:
metric_key: rising_star
expected_condition: YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'
generated_where: yearsatcompany <= 2 and performancerating = 4 and attrition = 'no'
is_correct: True
reward: 1.0
reason: SQL cumple exactamente con el glosario.

RED NEURONAL:
Predicted score before update: 0.612007737159729
Training info: {'pred_before_update': 0.612007737159729, 'loss': 0.4910103678703308, 'target_reward': 1.0}

RESULTADO:


,total
0,30


{'question': '¿Cuántos empleados son estrellas en ascenso?',
 'metric_key': 'rising_star',
 'used_memory': False,
 'planner_reasoning': "La pregunta se refiere directamente a empleados que son 'estrellas en ascenso', lo cual coincide exactamente con la métrica 'rising_star' del glosario.",
 'sql': "\n    SELECT COUNT(*) AS total\n    FROM employees\n    WHERE YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'\n    ",
 'result':    total
 0     30,
 'evaluation': {'metric_key': 'rising_star',
  'expected_condition': "YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'",
  'generated_where': "yearsatcompany <= 2 and performancerating = 4 and attrition = 'no'",
  'is_correct': True,
  'reward': 1.0,
  'reason': 'SQL cumple exactamente con el glosario.'},
 'predicted_score_before': 0.612007737159729,
 'training_info': {'pred_before_update': 0.612007737159729,
  'loss': 0.4910103678703308,
  'target_reward': 1.0}}

In [75]:
run_demo("¿Cuántos empleados son estrellas en ascenso?")

PREGUNTA:
¿Cuántos empleados son estrellas en ascenso?

MÉTRICA DETECTADA:
rising_star

PLANNER REASONING:
La pregunta menciona 'estrellas en ascenso', que corresponde directamente a la métrica 'rising_star' en el glosario.

¿USÓ MEMORIA?
True

SQL GENERADO:

    SELECT COUNT(*) AS total
    FROM employees
    WHERE YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'
    

EVALUACIÓN:
metric_key: rising_star
expected_condition: YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'
generated_where: yearsatcompany <= 2 and performancerating = 4 and attrition = 'no'
is_correct: True
reward: 1.0
reason: SQL cumple exactamente con el glosario.

RED NEURONAL:
Predicted score before update: 0.6376734972000122
Training info: {'pred_before_update': 0.6376734972000122, 'loss': 0.449928879737854, 'target_reward': 1.0}

RESULTADO:


,total
0,30


{'question': '¿Cuántos empleados son estrellas en ascenso?',
 'metric_key': 'rising_star',
 'used_memory': True,
 'planner_reasoning': "La pregunta menciona 'estrellas en ascenso', que corresponde directamente a la métrica 'rising_star' en el glosario.",
 'sql': "\n    SELECT COUNT(*) AS total\n    FROM employees\n    WHERE YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'\n    ",
 'result':    total
 0     30,
 'evaluation': {'metric_key': 'rising_star',
  'expected_condition': "YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'",
  'generated_where': "yearsatcompany <= 2 and performancerating = 4 and attrition = 'no'",
  'is_correct': True,
  'reward': 1.0,
  'reason': 'SQL cumple exactamente con el glosario.'},
 'predicted_score_before': 0.6376734972000122,
 'training_info': {'pred_before_update': 0.6376734972000122,
  'loss': 0.449928879737854,
  'target_reward': 1.0}}

In [69]:
print("Casos en memoria:", len(memory_cases))
print(memory_cases[-1] if memory_cases else "Sin casos")

Casos en memoria: 3
{'question': '¿Cuántos empleados son estrellas en ascenso?', 'metric_key': 'rising_star', 'sql': "\n    SELECT COUNT(*) AS total\n    FROM employees\n    WHERE YearsAtCompany <= 2 AND PerformanceRating = 4 AND Attrition = 'No'\n    ", 'reward': 1.0, 'result_preview': [{'total': 30}]}


In [70]:
print(heuristic_metric_from_question("¿Cuántos empleados son estrellas en ascenso?"))

rising_star


In [74]:
planner_output = run_planner("¿Cuántos empleados son estrellas en ascenso?")
print(planner_output)
print("use_memory =", planner_output.use_memory)

metric_key='rising_star' use_memory=True reasoning="La pregunta se refiere a empleados que son 'estrellas en ascenso', lo cual coincide exactamente con la métrica 'rising_star' del glosario."
use_memory = True
